# 6.1 Dimensionality reduction, and what it does to you

Every dataset so far had few enough columns to plot. This one does not: a handwritten digit
is 784 numbers, and there is no scatter plot of 784 dimensions. Two methods squash that into
a picture you can look at.

Both work. Both also have a decision inside them that they will not make for you, and both
produce a confident-looking picture either way:

- **PCA** finds the directions along which the data varies most — measured in whatever units
  you handed it. Give it grams and it will find grams.
- **t-SNE** arranges points so that neighbours stay neighbours. How many neighbours is a
  parameter, and the shape of the answer depends on it.

The notebook builds up from a sketch you can check by eye — what "the direction of most
variance" means, and why keeping it loses the least — to the two showcases where the
methods go wrong. The sections named "where it goes wrong" are staged: the failure is
deliberate and the correct version is right next to it.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import PlotSettings, ProjectionPlot, ScreePlot
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_openml, make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from wa_analyzer.data import load_showcase

rng = np.random.default_rng(42)


## 6.1.1 What PCA is looking for

Reducing three columns to two means choosing a flat surface and dropping every point onto
it; reducing two to one means choosing a *line*. Start with the line, because there you can
see the whole idea in one sketch.

Take a cloud of points with the mean subtracted, and any line through the origin. Each point
splits into two pieces: where it lands *along* the line, and how far it sits *from* the line.
By Pythagoras, `(distance to origin)² = (position along the line)² + (distance to the line)²`,
and the left-hand side does not depend on which line you chose. So summed over the cloud,
**the line with the most spread along it is exactly the line the points sit closest to.**
That is the whole of PCA in one sentence: the direction of greatest variance is also the
projection that loses the least. The sketch draws both halves — a geometric picture, not a
chart of data, so it is plain matplotlib.


Red is what the projection throws away; the black dots are what it keeps. Rotate the line
and the red segments grow exactly as fast as the black spread shrinks.


In [ ]:
cloud = rng.multivariate_normal([0, 0], [[3.0, 2.2], [2.2, 2.0]], size=40)
cloud -= cloud.mean(axis=0)

direction = np.linalg.svd(cloud)[2][0]  # the line with the most spread along it
along = cloud @ direction  # every point's position along that line
feet = np.outer(along, direction)  # where every point lands on it

fig, ax = plt.subplots(figsize=(6, 6))
reach = np.abs(along).max() * 1.1
ax.plot([-reach * direction[0], reach * direction[0]], [-reach * direction[1], reach * direction[1]],
        color="black", linewidth=1, label="the line PCA picks")
for point, foot in zip(cloud, feet):
    ax.plot([point[0], foot[0]], [point[1], foot[1]], color="crimson", linewidth=0.8, alpha=0.7)
ax.scatter(cloud[:, 0], cloud[:, 1], color="#4c72b0", zorder=3, label="data, centred")
ax.scatter(feet[:, 0], feet[:, 1], color="black", s=14, zorder=4, label="the same points, projected")
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc="upper left")
_ = ax.set_title("Most spread along the line = least distance to the line")


### Three columns that are really two

Now the same idea one dimension up, on data built so that the answer is known. A hundred
points on a curved strip: two columns trace the strip, and the third is a weighted sum of
the first two plus a little noise. Three numbers per row, but the rows all lie close to one
tilted plane — so a good two-dimensional picture should exist, and PCA should find it.


In [ ]:
def tilted_sheet(rng: np.random.Generator, m: int = 100, noise: float = 0.1) -> np.ndarray:
    """m points on a curved strip; the third column is 0.1 * first + 0.3 * second, plus noise."""
    angles = rng.random(m) * 3 * np.pi / 2 - 0.5
    sheet = np.empty((m, 3))
    sheet[:, 0] = np.cos(angles) + np.sin(angles) / 2 + noise * rng.standard_normal(m) / 2
    sheet[:, 1] = np.sin(angles) * 0.7 + noise * rng.standard_normal(m) / 2
    sheet[:, 2] = sheet[:, 0] * 0.1 + sheet[:, 1] * 0.3 + noise * rng.standard_normal(m)
    return sheet


sheet = tilted_sheet(np.random.default_rng(4))


### The three pieces of an SVD

`np.linalg.svd` on the centred data returns three pieces, and together they are the
matrix you started with:

$$X = U \Sigma V^T$$

- The rows of **$V^T$** are the directions — the line from the sketch first, then the best
  line at right angles to it, then the one at right angles to both. Three columns in, three
  directions out.
- **$\Sigma$** holds one number per direction, sorted largest first. Squared, each is the
  spread the data has along that direction, so `s² / sum(s²)` is the share of the variance
  each direction carries.
- **$U$** says where every row lands along each direction — the "position along the line"
  from the sketch, once per direction.

Three words describe what makes the rows of $V^T$ usable as axes. They are **orthogonal**:
at right angles, so what one direction measures the next one cannot measure again.
**Normalised**: each has length one, so a coordinate along it is in the data's own units.
And together they are a **basis**: any point in the data can be written as so much of the
first direction plus so much of the second plus so much of the third, and nothing is lost
until you drop one. Dropping the last one is the projection.


In [ ]:
sheet_centred = sheet - sheet.mean(axis=0)
U, s, Vt = np.linalg.svd(sheet_centred)
U.shape, s.shape, Vt.shape


The data and the three directions, with each arrow's length showing its singular value
relative to the largest:


In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(sheet_centred[:, 0], sheet_centred[:, 1], sheet_centred[:, 2])

for i, colour in enumerate(["r", "g", "b"]):
    direction = Vt[i] * s[i] / s[0]  # arrow length: this direction's share of the spread
    ax.quiver(0, 0, 0, *direction, color=colour, arrow_length_ratio=0.1, linewidths=3)

ax.set_xlabel("column 1")
ax.set_ylabel("column 2")
ax.set_zlabel("column 3")  # ty: ignore[unresolved-attribute] -- Axes3D, not the base Axes the stub sees
_ = ax.set_title("Three directions, arrow length = relative singular value")


The red and green arrows lie in the tilted plane the points sit on; the blue one is nearly
invisible, because it points straight out of that plane and the only spread in that
direction is the noise. None of the three is a column of the data — the red arrow is a
mixture of all three columns — which is what "PCA finds new axes" means.


Keep the first two directions and drop the third: `sheet_centred @ Vt.T[:, :2]` is the
position of every point along the red and green arrows, and nothing else.


In [ ]:
W2 = Vt.T[:, :2]
sheet_2d_svd = sheet_centred @ W2

settings = PlotSettings(
    figsize=(6, 5),
    title="The sheet, projected onto its first two directions",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=sheet_2d_svd, color="#4c72b0")


The curved strip, seen flat on: the picture a plane's worth of data deserves. That is the
whole of PCA, and `sklearn`'s `PCA` does exactly this — centre, SVD, keep the first
`n_components` directions:


In [ ]:
pca = PCA(n_components=2)
sheet_2d = pca.fit_transform(sheet)
np.allclose(sheet_2d_svd, sheet_2d) or np.allclose(sheet_2d_svd, -sheet_2d)


The same coordinates, up to a sign: a direction and its mirror image describe the same line,
so an axis in a PCA plot may come out flipped between two runs or two libraries. The
picture means the same thing either way.


`explained_variance_ratio_` is the `s² / sum(s²)` from above, and `ScreePlot` is the
picture of it. Read it before reading any projection: it says how much of the data the
projection is a picture of.


In [ ]:
pca = PCA().fit(sheet)
print("explained variance ratio per component:")
print(np.round(pca.explained_variance_ratio_, 4))
print("\nfrom the singular values, squared and normalised:")
print(np.round(np.square(s) / np.sum(np.square(s)), 4))

settings = PlotSettings(
    figsize=(6, 4),
    title=f"Two components carry {pca.explained_variance_ratio_[:2].sum():.1%} of the variance",
    xlabel="component",
    ylabel="share of variance",
)
fig, ax = ScreePlot(settings).plot(explained_variance_ratio=pca.explained_variance_ratio_)


## 6.1.2 Where PCA goes wrong: the units you gave it

PCA maximises variance. Variance has units — squared ones — so the component it finds is
partly a fact about the data and partly a fact about what you measured things in. On the
sheet above that was invisible, because all three columns were the same made-up scale.

The penguins from lesson 2 are not. Look at the standard deviations before doing anything
else.


In [ ]:
penguins = load_showcase("penguins").dropna()
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(penguins[FEATURES].describe().loc[["count", "mean", "std"]].round(1).to_string())

In [ ]:
measurements = penguins[FEATURES].to_numpy()
unscaled = PCA().fit(measurements)

print(f"explained variance ratio: {np.round(unscaled.explained_variance_ratio_, 4)}")
print("\nwhat the first component is made of:")
print(pd.Series(unscaled.components_[0], index=FEATURES).round(3).to_string())

**The first component explains 99.99% of the variance and it is body mass**, with a loading
of 1.000 against roughly zero for everything else. PCA did not fail — it did exactly what it
promises. Body mass has a standard deviation of 805, bill depth has 2.0, so essentially all
the variance in this matrix is grams, and the direction of greatest variance is the grams
axis.

A 99.99% number in a report looks like a triumph. It is the symptom.

The clearest way to see that this is about units and not about penguins: divide body mass by
a thousand and call it kilograms. Same birds, same measurements, same information.

In [ ]:
kilos = measurements.copy()
kilos[:, FEATURES.index("body_mass_g")] /= 1000
in_kilos = PCA().fit(kilos)

print(f"explained variance ratio: {np.round(in_kilos.explained_variance_ratio_, 4)}")
print("\nwhat the first component is made of now:")
print(pd.Series(in_kilos.components_[0], index=FEATURES).round(3).to_string())

The first component is now **flipper length**, at a loading of 0.96, and it explains 91.9%
instead of 99.99%. Nothing about the birds changed. A unit did.

So before PCA, put every column on the same footing: subtract the mean and divide by the
standard deviation, which is `StandardScaler`. After that a "unit" of every column is one
standard deviation of that column, and no column can win by being measured in something small.

In [ ]:
standardised = StandardScaler().fit_transform(measurements)
scaled = PCA().fit(standardised)

print(f"explained variance ratio: {np.round(scaled.explained_variance_ratio_, 4)}\n")
loadings = pd.DataFrame(scaled.components_[:2].T, index=pd.Index(FEATURES), columns=pd.Index(["PC1", "PC2"]))
print(loadings.round(2).to_string())

Now the components mean something you can say out loud.

**PC1 is size**: bill length, flipper length and body mass all load positively, so a penguin
high on PC1 is a big penguin. Bill *depth* loads negatively — which is 5.2's finding
arriving from a different direction, because the long-billed species is the one with shallow
bills.

**PC2 is bill shape**: length and depth together, flipper and mass at nothing. It is the
"how chunky is the beak" axis, and it is a real second thing about a penguin rather than a
rounding error on the first.

Two components, 88% of the variance, and a sentence per axis. Compare that with the
99.99% version, which had one axis and it was a scale.

In [ ]:
def separation(scores: np.ndarray) -> float:
    """Between-species variance over within-species variance, along one component."""
    frame = pd.DataFrame({"score": scores, "species": penguins.species.to_numpy()})
    return frame.groupby("species").score.mean().var() / frame.groupby("species").score.var().mean()


print(f"species separation along PC1, unscaled: "
      f"{separation(unscaled.transform(measurements)[:, 0]):.2f}")
print(f"species separation along PC1, scaled:   "
      f"{separation(scaled.transform(standardised)[:, 0]):.2f}")

settings = PlotSettings(
    figsize=(11, 4.5),  # ty: ignore[invalid-argument-type]
    title="The same 333 penguins, before and after scaling",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[
        f"raw units — PC1+PC2 = {unscaled.explained_variance_ratio_[:2].sum():.1%}",
        f"standardised — PC1+PC2 = {scaled.explained_variance_ratio_[:2].sum():.1%}",
    ],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=2)
for ax, model, values in [(axes[0], unscaled, measurements), (axes[1], scaled, standardised)]:
    host.plot_on_axes(ProjectionPlot(settings), ax,
                      coordinates=model.transform(values)[:, :2],
                      labels=penguins.species.to_numpy(), legend=(ax is axes[1]))

It is not only a matter of interpretation: the unscaled version is **worse at the job**. Along
PC1, the ratio of between-species to within-species variance goes from 3.09 to 9.50 once the
columns are standardised. On the left, Gentoo pulls away — they are the heavy ones — but
Adelie and Chinstrap stay thoroughly mixed, because the only thing separating them is bill
shape and bill shape is measured in millimetres. On the right all three come apart.

And look at what the left panel does not tell you. Its two axes are 99.99% and 0.01% of the
variance, so honestly drawn it would be a horizontal line; matplotlib stretched the second
axis to fill the panel, and a 0.01% component now looks like half the picture. **A projection
plot always fills its frame.** That is exactly why the scree plot is not optional.

> **The rule.** Standardise before PCA unless every column is already in the same unit *and*
> you mean for the bigger-varying ones to count more. Pixel intensities, all 0–255, are the
> usual exception — which is why the MNIST section below does not scale.

## 6.1.3 The swiss roll: a shadow, or a map of neighbourhoods

Scaling fixes what PCA measures; it does not change what PCA *is*, which is a flat shadow.
The swiss roll is the standard dataset for seeing the limit of that: a two-dimensional
sheet, rolled up inside three dimensions. Colour is the position along the sheet — the one
thing a picture of this data should preserve.


In [ ]:
roll, colour = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(roll[:, 0], roll[:, 1], roll[:, 2], c=colour, cmap=plt.cm.hot)  # ty: ignore[unresolved-attribute]
ax.view_init(10, -70)  # ty: ignore[unresolved-attribute] -- Axes3D, not the base Axes the stub sees
_ = ax.set_title("A two-dimensional sheet, rolled up in three dimensions")


PCA first. It keeps 71% of the variance, which sounds like most of the data, and the picture
is a spiral seen from the side.


In [ ]:
pca = PCA(n_components=2)
roll_2d = pca.fit_transform(roll)

settings = PlotSettings(
    figsize=(6, 5),
    title=f"Swiss roll under PCA — {pca.explained_variance_ratio_.sum():.0%} of the variance",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=roll_2d, labels=colour, palette="hot",
                                        legend=False)

Look at where the colours land. Yellow is one end of the sheet and black is the other, as
far apart along the sheet as two points can be — and in the shadow they sit right next to
each other, because the roll's turns overlap from every flat viewpoint. The shadow is
faithful to the 3D geometry and useless for the question "which points are near each other
*on the sheet*".

t-SNE asks that question directly. Instead of a viewpoint, it looks for a 2D arrangement
in which each point's nearest neighbours in 3D are still its nearest neighbours.


In [ ]:
tsne = TSNE(n_components=2, random_state=42)
roll_tsne = tsne.fit_transform(roll)

settings = PlotSettings(
    figsize=(6, 5),
    title="Swiss roll under t-SNE",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=roll_tsne, labels=colour, palette="hot",
                                        legend=False)


The colour now runs from one end to the other: black at one side, yellow at the other, the
sheet more or less unrolled. That is the thing PCA could not do, because no flat viewpoint
undoes a curve. Look at what it cost, though. The sheet is torn into clumps with gaps
between them, and those gaps are not in the data — the roll is one continuous strip. t-SNE
kept every point's neighbours and gave up everything else: how far apart two clumps sit, and
whether a gap is real, are things this picture cannot tell you.

That is the trade in one sentence: **PCA keeps the global geometry and cannot undo a curve;
t-SNE undoes the curve and gives up the global geometry.**

### The manifold hypothesis

The roll is a two-dimensional thing that happens to live in three dimensions. The bet
behind using t-SNE on real data is that the same is true at scale: that 784 pixel values of
a handwritten digit do not really spread over 784 independent directions, but sit on some
curved, much lower-dimensional surface inside that space — a *manifold* — because a "7" can
only vary in so many ways. If that holds, a picture that unrolls the surface should show
ten digits as ten groups, and a flat shadow should not. Check it on the digits themselves.

## 6.1.4 MNIST: 784 pixels

The download is 70,000 images and goes to `~/.cache/mads-dav` once.


In [ ]:
cache = Path.home() / ".cache/mads-dav"
cache.mkdir(parents=True, exist_ok=True)

mnist = fetch_openml("mnist_784", version=1, as_frame=False, data_home=str(cache))
mnist.target = mnist.target.astype(np.uint8)


In [ ]:
digits = mnist["data"]
digit_labels = mnist["target"]
print(f"{digits.shape[0]:,} digits, each one {digits.shape[1]} pixels")

Seventy thousand digits is more than t-SNE wants: it is roughly quadratic in the number of
points, and this notebook fits it six times. Five thousand, drawn at random, is enough to see
everything below and turns a coffee break into a few seconds.

That is a decision too. Write it down when you make it — a picture of a sample is a picture of
a sample.

In [ ]:
rng = np.random.default_rng(42)
sample = rng.choice(len(digits), size=5000, replace=False)
subset, subset_labels = digits[sample], digit_labels[sample]
print(f"{subset.shape[0]:,} digits, {subset.shape[1]} pixels each")

Every digit is a 28x28 image flattened into a row of 784 numbers, one per pixel. Here is the
first one, folded back into a square:

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(digits[0].reshape(28, 28), cmap="gray")
ax.set_xticks([])
ax.set_yticks([])
_ = ax.set_title(f"digit {digit_labels[0]}, as 784 numbers")


A row of 784 pixel intensities, all on the same 0–255 scale — which is the one case where
not standardising before PCA is a defensible choice: every column already shares a unit, and
a pixel that varies more across digits *should* count more. First the shadow, with its scree
plot next to it.


In [ ]:
pca = PCA().fit(subset)
cumulative = np.cumsum(pca.explained_variance_ratio_)

print(f"the first two components carry {cumulative[1]:.1%} of the variance")
for target in (0.5, 0.8, 0.95):
    print(f"  components needed for {target:.0%}: {int(np.argmax(cumulative >= target)) + 1}")

settings = PlotSettings(
    figsize=(7, 4),
    title="784 pixels, and how the variance is spread over them",
    xlabel="component",
    ylabel="share of variance",
)
fig, ax = ScreePlot(settings).plot(explained_variance_ratio=pca.explained_variance_ratio_,
                                   n_components=20)

In [ ]:
settings = PlotSettings(
    figsize=(7, 6),
    title=f"MNIST under PCA — two components, {cumulative[1]:.1%} of the variance",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=pca.transform(subset)[:, :2],
                                        labels=subset_labels, palette="tab10", s=8)

The digits overlap into a single smear, and the scree plot says why: **two components carry
17.0% of the variance**, and you need 148 of them for 95%. The picture is not wrong; it is a
picture of 17% of this dataset, and that number belongs in the title — which is what
`ScreePlot` and the title above it are for.

This is the honest version of "PCA does something but it is not very useful in two
dimensions". It is not useless; it is showing you the 17% that fits.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, init="pca")
mnist_tsne = tsne.fit_transform(subset)

settings = PlotSettings(
    figsize=(7, 6),
    title="MNIST under t-SNE, default perplexity of 30",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=mnist_tsne, labels=subset_labels,
                                        palette="tab10", s=8)

Ten clumps, one per digit, and the pairs that touch are the pairs that look alike. This is
t-SNE doing real work that PCA could not: the manifold is curved, and PCA can only cut it
with a flat plane. The bet from §6.1.3 paid off.

Now the parameter.

## 6.1.5 Where t-SNE goes wrong: perplexity is not a rendering option

Perplexity is roughly "how many neighbours each point should try to stay near". It has a
default of 30, and defaults are where decisions go to hide. Same 5,000 digits, four values.


In [ ]:
PERPLEXITIES = (5, 30, 100, 200)
embeddings = {
    perplexity: TSNE(n_components=2, perplexity=perplexity, random_state=42,
                     init="pca").fit_transform(subset)
    for perplexity in PERPLEXITIES
}

settings = PlotSettings(
    figsize=(11, 9),
    title="One dataset, four perplexities",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[f"perplexity {perplexity}" for perplexity in PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(PERPLEXITIES))
for ax, perplexity in zip(axes, PERPLEXITIES):
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embeddings[perplexity],
                      labels=subset_labels, palette="tab10", legend=False, s=8)

In [ ]:
pairs = np.triu_indices(10, k=1)
gaps = {}
for perplexity, embedding in embeddings.items():
    frame = pd.DataFrame(embedding, columns=pd.Index(["x", "y"])).assign(digit=subset_labels)
    centroids = frame.groupby("digit")[["x", "y"]].mean().to_numpy()
    distances = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=-1)
    gaps[perplexity] = distances[pairs] / distances[pairs].max()
    print(f"perplexity {perplexity:>3d}: silhouette by true digit "
          f"{silhouette_score(embedding, subset_labels):.2f}")

print("\nspearman between the pairwise gaps each perplexity draws:")
table = pd.DataFrame(
    [[spearmanr(gaps[a], gaps[b]).statistic for b in PERPLEXITIES] for a in PERPLEXITIES],
    index=pd.Index(PERPLEXITIES), columns=pd.Index(PERPLEXITIES),
)
print(table.round(2).to_string())

names = [f"{a}-{b}" for a in range(10) for b in range(a + 1, 10)]
print("\nwhich digits the picture puts closest together, and furthest apart:")
for perplexity in (PERPLEXITIES[0], PERPLEXITIES[-1]):
    order = np.argsort(gaps[perplexity])
    print(f"  perplexity {perplexity:>3d}: closest {[names[i] for i in order[:3]]}, "
          f"furthest {[names[i] for i in order[-3:]]}")


Two different answers, and you need both.

**Membership is stable.** The silhouette scored against the true digit stays between 0.20 and
0.28 in all four. Whichever perplexity you pick, the ten groups come out as ten groups. When
you say "there are clusters here", t-SNE is backing you up.

**Geometry is not.** The gaps between those clusters, ranked, agree between perplexity 5 and
100 at a spearman of only 0.41. At perplexity 5 the two digits furthest apart are 0 and 1;
at perplexity 200 they are 0 and 7. So *how far apart* two clumps sit is a fact about the
parameter, not about handwriting.

The one thing that survives is 4-9 as the closest pair at both extremes, which is the
confusion any person would predict. A relationship that holds across settings is worth
reporting. The arrangement of a single picture is not.

> **What you may say about a t-SNE plot:** which points ended up together, and whether that
> survives a change of perplexity. **What you may not say:** that one cluster is nearer to a
> second than to a third, that a gap is wide, that a cluster is large, or anything at all
> about the axes — which is why `ProjectionPlot` hides the ticks.

## 6.1.6 The failure that costs people papers

Everything above had real structure in it — ten digits genuinely are ten things. Watch what
the same tool does to data with no structure whatsoever: a thousand points, fifty dimensions,
every value an independent draw from the same normal distribution. There are no groups. There
is nothing to find.


In [ ]:
noise = np.random.default_rng(0).normal(size=(1000, 50))

NOISE_PERPLEXITIES = (2, 5, 30, 100)
settings = PlotSettings(
    figsize=(11, 9),
    title="Fifty dimensions of pure noise, four perplexities",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[f"perplexity {perplexity}" for perplexity in NOISE_PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(NOISE_PERPLEXITIES))

scores = {}
for ax, perplexity in zip(axes, NOISE_PERPLEXITIES):
    embedding = TSNE(n_components=2, perplexity=perplexity, random_state=0,
                     init="pca").fit_transform(noise)
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embedding, color="#777777")
    scores[perplexity] = silhouette_score(
        embedding, KMeans(5, n_init=10, random_state=0).fit_predict(embedding))


In [ ]:
raw_score = silhouette_score(noise, KMeans(5, n_init=10, random_state=0).fit_predict(noise))
print(f"k-means with k=5 on the raw 50-dimensional noise: silhouette {raw_score:.3f}")
for perplexity, score in scores.items():
    print(f"  ...on the t-SNE picture at perplexity {perplexity:>3d}: silhouette {score:.3f}")

At perplexity 2 the picture visibly breaks into little clumps. There are no clumps. The
algorithm was asked to keep each point near its two nearest neighbours, and in fifty
dimensions of noise every point has two nearest neighbours, so it obliged.

The numbers say something worse, and it holds at *every* perplexity. Run k-means on the raw
data and the silhouette is **0.016** — correctly reporting that there is nothing there. Run
the identical k-means on the two-dimensional picture and it is around **0.31**, which in any
other context you would read as decent cluster structure.

**So never cluster the embedding.** Distances in a t-SNE plot are not the data's distances;
the projection has squeezed fifty dimensions into a bounded patch of paper, and anything
measuring distance there is measuring the paper. Cluster the data, then use the projection to
look at what you found.

## What to write down

1. **Did you scale, and why.** "All columns were pixels" is a reason. "It ran either way" is
   not.
2. **The variance your picture contains** — from the scree plot, in the caption. Two
   components of 784 is a number a reader cannot guess.
3. **The perplexity**, and one other perplexity you looked at. If the finding only exists at
   one setting, it is a finding about the setting.
4. **Which claim you are making**: that points group together, or where those groups sit
   relative to each other. The first can survive. The second cannot.

---

**Where this goes next.** 06.2 keeps the digits and asks a different question — not "what
does this look like" but "can a model tell them apart", which is the check that a shape you
saw in a projection was worth believing.